# Ben Soliman Mapping Pipeline
Maps Ben Soliman products to MaxAB products using Arabic NLP, brand matching, category mapping, size validation, and price comparison.

In [ ]:
import pandas as pd
import numpy as np
import re
import os
import warnings
from rapidfuzz import fuzz, process
from collections import Counter
from datetime import datetime
from common_functions import upload_dataframe_to_snowflake, send_file_slack, google_sheets
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 25)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.width', 220)
print("Imports loaded.")

In [ ]:
import sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))
import setup_environment_2
setup_environment_2.initialize_env()
from db import query_snowflake

# Load BS data from Snowflake
BS_QUERY = '''
SELECT name AS BEN_SOLIMAN_SKU, category AS CATEGORY,
       package_unit AS BS_PACKAGE_UNIT, BASIC_UNIT_COUNT AS BS_BASIC_UNIT_COUNT,
       price AS BS_PRICE
FROM (
    SELECT *, MAX(inject_date) OVER (PARTITION BY name) AS last_date
    FROM MATERIALIZED_VIEWS.SAVVY_RAW
    WHERE inject_date >= CURRENT_DATE - 1
    QUALIFY last_date = inject_date
)
'''
print('Loading Ben Soliman data from Snowflake...')
bs = query_snowflake(BS_QUERY)
bs.columns = bs.columns.str.upper().str.strip()
print(f'BS raw: {bs.shape}')
print(bs.columns.tolist())
print(bs.head(2))

# Load MaxAB data from SQL file
print('\nLoading MaxAB data from Snowflake...')
with open('current_sku_data.sql', 'r', encoding='utf-8') as f:
    MAXAB_QUERY = f.read()
maxab = query_snowflake(MAXAB_QUERY)
maxab.columns = maxab.columns.str.upper().str.strip()
print(f'MaxAB raw: {maxab.shape}')
print(maxab.columns.tolist())
print(maxab.head(2))

# Savvy mapping for comparison
SAVVY_QUERY = '''
SELECT query_id AS MAXAB_PRODUCT_ID, app AS SOURCE_APP,
       match AS BS_DESC
FROM materialized_views.competitors_mapping_fixed
'''
print('\nLoading Savvy mapping from Snowflake...')
savvy = query_snowflake(SAVVY_QUERY)
savvy.columns = savvy.columns.str.upper().str.strip()
print(f'Savvy mapping: {savvy.shape}')

In [ ]:
ARABIC_DIACRITICS = re.compile(r'[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06DC\u06DF-\u06E4\u06E7\u06E8\u06EA-\u06ED]')
ALEF_VARIANTS = {'\u0622': '\u0627', '\u0623': '\u0627', '\u0625': '\u0627', '\u0671': '\u0627'}
MEASUREMENT_NORMALIZATION = {
    'جرام': 'جم', 'غرام': 'جم', 'غم': 'جم', 'جرم': 'جم', 'g': 'جم', 'gm': 'جم', 'gr': 'جم',
    'كيلو': 'كجم', 'كيلوجرام': 'كجم', 'كيلوغرام': 'كجم', 'kg': 'كجم',
    'جنيه': 'جنيه', 'جنية': 'جنيه', 'ج': 'جنيه',
    'ملل': 'مل', 'ملي': 'مل', 'ميلي': 'مل', 'ml': 'مل',
    'لتر': 'لتر', 'ليتر': 'لتر', 'l': 'لتر', 'سنتي': 'سم', 'سنتيمتر': 'سم', 'cm': 'سم',
    'قطعه': 'قطعة', 'حبه': 'حبة', 'عبوه': 'عبوة', 'فتله': 'فتلة', 'بكره': 'بكرة', 'حفاضه': 'حفاضة',
}
SIZE_TO_BASE = {'جم': ('weight', 1.0), 'كجم': ('weight', 1000.0), 'مل': ('volume', 1.0), 'لتر': ('volume', 1000.0), 'سم': ('length', 10.0)}
COUNT_UNITS = set(['فتله','فتلة','بكره','بكرة','قطعه','قطعة','حبه','حبة','كيس','ظرف','رول',
                   'باكت','منديل','فوطه','فوطة','شريط','قرص','ورقه','ورقة','كبسوله','كبسولة','باكيت',
                   'جنيه'])
EASTERN_TO_WESTERN = str.maketrans('٠١٢٣٤٥٦٧٨٩', '0123456789')

def normalize_arabic(t):
    if not isinstance(t, str) or not t.strip(): return ''
    t = ARABIC_DIACRITICS.sub('', t.strip()).replace('\u0640', '')
    for v, c in ALEF_VARIANTS.items(): t = t.replace(v, c)
    t = t.replace('\u0629', '\u0647').replace('\u0649', '\u064A').translate(EASTERN_TO_WESTERN).lower()
    t = re.sub(r'[^\w\s\u0600-\u06FF]', ' ', t)
    t = re.sub(r'(\d)([\u0600-\u06FF])', r'\1 \2', t)
    t = re.sub(r'([\u0600-\u06FF])(\d)', r'\1 \2', t)
    tokens = [MEASUREMENT_NORMALIZATION.get(x, x) for x in t.split()]
    return re.sub(r'\s+', ' ', ' '.join(tokens)).strip()

def strip_al(token):
    if token.startswith('ال') and len(token) > 2: return token[2:]
    return token

def extract_size_info(t): return re.findall(r'(\d+(?:\.\d+)?)\s*(كجم|جم|مل|لتر|سم)', normalize_arabic(t))
def normalize_count_unit(u): return u.replace('\u0629', '\u0647')
def extract_count_info(t):
    normalized = normalize_arabic(t)
    matches = []; seen = set()
    for unit in COUNT_UNITS:
        for m in re.finditer(r'(\d+)\s*' + re.escape(unit), normalized):
            key = (m.group(1), normalize_count_unit(unit))
            if key not in seen: matches.append((float(m.group(1)), normalize_count_unit(unit))); seen.add(key)
        for m in re.finditer(re.escape(unit) + r'\s*(\d+)', normalized):
            key = (m.group(1), normalize_count_unit(unit))
            if key not in seen: matches.append((float(m.group(1)), normalize_count_unit(unit))); seen.add(key)
    return matches

def normalize_size_to_base(v, u):
    if u in SIZE_TO_BASE: d, m = SIZE_TO_BASE[u]; return d, float(v) * m
    return None, None

def sizes_compatible(a, b, tol=0.15):
    sa, sb = extract_size_info(a), extract_size_info(b)
    ca, cb = extract_count_info(a), extract_count_info(b)
    if sa and sb:
        for va, ua in sa:
            da, ba = normalize_size_to_base(va, ua)
            if da is None: continue
            for vb, ub in sb:
                db, bb = normalize_size_to_base(vb, ub)
                if db is None: continue
                if da != db: return False
                mx = max(ba, bb)
                if mx > 0: return abs(ba - bb) / mx <= tol
                return True
    if ca and cb:
        for val_a, unit_a in ca:
            for val_b, unit_b in cb:
                if unit_a == unit_b:
                    mx = max(val_a, val_b)
                    if mx > 0: return abs(val_a - val_b) / mx <= tol
                    return True
    has_size_a, has_size_b = bool(sa), bool(sb)
    has_count_a, has_count_b = bool(ca), bool(cb)
    if (has_size_a and has_count_b and not has_size_b) or (has_count_a and has_size_b and not has_count_b):
        return False
    return None

def text_sim(a, b):
    if not a or not b: return 0.0
    return max(fuzz.token_sort_ratio(a, b), fuzz.token_set_ratio(a, b) * 0.95, fuzz.partial_ratio(a, b) * 0.85)

def pct_diff(a, b):
    mn = min(a, b)
    if mn <= 0: return 1.0
    return (max(a, b) - mn) / mn

CATEGORY_WORDS = {normalize_arabic(w) for w in [
    'مكرونه','بسكويت','جبنه','حفاضات','مناديل','بن','مربي','كريم','لبان','ارز',
    'ملح','تونه','صابون','جيل','مسحوق','سائل','شامبو','شاي','عصير','زيت','لبن','زبادي',
    'زبده','سمن','سمنه','دقيق','سكر','شوكولاته','شيكولاته','كاكاو','قهوه',
    'شيبسي','ويفر','كيك','نودلز','صلصه','صوص','كاتشب','مايونيز','توابل',
    'عسل','حلاوه','طحينه','فول','حمص','سردين','ماكريل','برجر','لانشون',
    'مشروب','مياه','منظف','منعم','مزيل','معجون','شاور','رول','كيس',
    'حلوي','بقسماط','نشا','خميره','فانيليا','مرقه','جيلي','تمر','شطه',
    'مبيد','كرواسون','توفي','بسطرمه','فويل','اكياس','قمامه',
    'حليب','معطر','مبيض','فوطه','سفره','كورن','عدس','لوبيا','كابتشينو',
    'بانتس','حفاضه','غسيل','اطباق','بشره','شعر','جسم','وجه',
]}

print("Arabic NLP core loaded.")

In [ ]:
bs['sku_norm'] = bs['BEN_SOLIMAN_SKU'].apply(normalize_arabic)
bs['cat_norm'] = bs['CATEGORY'].apply(normalize_arabic)

bs['unit_price'] = bs['BS_PRICE'] / bs['BS_BASIC_UNIT_COUNT']

maxab['sku_norm'] = maxab['SKU'].apply(normalize_arabic)
maxab['brand_norm'] = maxab['BRAND'].apply(normalize_arabic)
maxab['cat_norm'] = maxab['CAT'].apply(normalize_arabic)

maxab['single_unit_price'] = maxab['CURRENT_PRICE'] / maxab['CHILD_QUANTITY']

brand_to_products = maxab.groupby('brand_norm')['PRODUCT_ID'].apply(set).to_dict()
maxab_products = maxab.drop_duplicates(subset='PRODUCT_ID')[['PRODUCT_ID','SKU','BRAND','CAT','sku_norm','brand_norm','cat_norm','CURRENT_PRICE','BASIC_UNIT_COUNT','CHILD_QUANTITY','single_unit_price']].copy().set_index('PRODUCT_ID')

base_brand_count = len(brand_to_products)
print(f"BS: {len(bs)} SKUs")
print(f"MaxAB: {maxab['PRODUCT_ID'].nunique()} products, {base_brand_count} brands (from BRAND column)")

# --- Source 1: Manual brand aliases from Google Sheet ---
print("\nLoading manual brand aliases from Google Sheet...")
try:
    brand_aliases = google_sheets('Unmatched Brands', 'Brands', 'get')
    brand_aliases.columns = brand_aliases.columns.str.strip()
    alias_count = 0
    for _, row in brand_aliases.iterrows():
        mx_brand = normalize_arabic(str(row['MaxAB Brand']))
        bs_brand = normalize_arabic(str(row['BS Brand']))
        if not mx_brand or not bs_brand:
            continue
        if mx_brand in brand_to_products:
            brand_to_products.setdefault(bs_brand, set()).update(brand_to_products[mx_brand])
            alias_count += 1
    print(f"  Loaded {alias_count} manual brand aliases")
except Exception as e:
    print(f"  Warning: Could not load brand aliases sheet: {e}")

# --- Source 2: Extract brand prefixes from MaxAB SKU names ---
_stop_tokens = CATEGORY_WORDS | set(MEASUREMENT_NORMALIZATION.values()) | COUNT_UNITS

def extract_sku_brand_prefix(sku_norm):
    tokens = sku_norm.split()
    prefix = []
    for t in tokens:
        if t in _stop_tokens or t.isdigit() or re.match(r'^\d', t):
            break
        prefix.append(t)
    return ' '.join(prefix) if prefix else None

sku_prefix_count = 0
for pid, row in maxab_products.iterrows():
    prefix = extract_sku_brand_prefix(row['sku_norm'])
    if prefix and len(prefix) >= 3 and prefix not in brand_to_products:
        brand_to_products.setdefault(prefix, set()).add(pid)
        sku_prefix_count += 1
    elif prefix and len(prefix) >= 3 and prefix in brand_to_products:
        brand_to_products[prefix].add(pid)

print(f"  Extracted {sku_prefix_count} new brand prefixes from SKU names")

# --- Rebuild brand vocabulary from merged keys ---
maxab_brands = sorted(brand_to_products.keys())
maxab_brand_set = set(maxab_brands)
maxab_brands_by_length = sorted(maxab_brands, key=len, reverse=True)

print(f"\nExpanded brand vocabulary: {base_brand_count} → {len(maxab_brands)} entries")

In [ ]:
CATEGORY_MAP = {
    'شيكولاتة': ['شوكولاتة', 'شوكولاتة سبريد', 'ويفر'],
    'لبان و حلويات': ['حلويات و لبان'],
    'عناية المنزل': ['مطهرات و ملمعات', 'سائل أطباق', 'معطرات'],
    'اسماك معلبه': ['تونة و سمك'],
    'منظفات ملابس': ['منظفات'],
    'أجبان': ['جبن', 'جبن سايب', 'جبنة موتزاريلا'],
    'البان': ['ألبان'],
    'بسكويت': ['بسكويت و معمول', 'ويفر'],
    'بقوليات': ['بقوليات'],
    'توابل': ['توابل وبهارات'],
    'حفاظات و فوط صحية': ['حفاضات أطفال', 'الفوط الصحية'],
    'حلاوه و طحينة': ['حلاوة طحينية', 'طحينة'],
    'حلويات سريعة التحضير': ['حلويات سريعة التحضير'],
    'زيوت': ['زيوت'],
    'سكر و دقيق و ارز': ['سكر', 'دقيق', 'أرز'],
    'سمنات': ['سمنة', 'زبدة'],
    'شاي و بن': ['شاي', 'قهوة'],
    'شعرية سريعة التحضير': ['شعرية سريعة التحضير'],
    'صابون أيدي': ['صابون'],
    'صابون سائل': ['صابون', 'سائل أطباق'],
    'صلصة و صوصات': ['صلصة و صوص'],
    'عسل و مربى': ['عسل', 'مربي'],
    'عصائر': ['عصاير', 'حاجه ساقعه'],
    'عناية الشخصية': ['شامبو و شاور جيل', 'كريم و جل للشعر', 'مزيل عرق', 'معجون أسنان'],
    'قهوة سريعة التحضير': ['قهوة', 'كاكاو و مبيضات'],
    'كيك و كرواسون': ['كيك وكرواسون'],
    'مرقة و خلطات': ['مرقة وخلطات'],
    'مشروبات غازية': ['حاجه ساقعه'],
    'مشروبات ساخنة': ['اعشاب', 'شاي'],
    'مشروبات شعير': ['حاجه ساقعه'],
    'مشروبات طاقة': ['مشروبات الطاقة'],
    'مقرمشات': ['مقرمشات', 'شيبسي'],
    'مكرونة': ['مكرونة'],
    'ورقيات': ['ورقيات'],
    'ميــاه معدنية': ['مياه معدنيه'],
    'أغذية معلبة': ['معلبات'],
    'خــل و ملــح': ['خل'],
    'سايب': ['جبن سايب'],
    'مستهلكات و بطاريات': ['بطاريات ولمبات', 'مستلزمات متعددة الاستخدمات واكياس'],
}

cat_map_norm = {}
for bs_cat, mx_cats in CATEGORY_MAP.items():
    bs_norm = normalize_arabic(bs_cat)
    cat_map_norm[bs_norm] = [normalize_arabic(c) for c in mx_cats]

def category_compatible(bs_cat_norm, maxab_cat_norm):
    """Check if BS category maps to MaxAB category."""
    if not bs_cat_norm or not maxab_cat_norm:
        return True
    mapped = cat_map_norm.get(bs_cat_norm, [])
    if mapped:
        return maxab_cat_norm in mapped
    return fuzz.ratio(bs_cat_norm, maxab_cat_norm) >= 70

mapped_count = sum(1 for c in bs['cat_norm'].unique() if c in cat_map_norm)
print(f"Category mapping: {mapped_count}/{bs['cat_norm'].nunique()} BS categories mapped")

In [ ]:
STOPWORDS = {'و','في','من','مع','بال','عرض','زياده','مجانا','هديه','مسعر',
             'جم','كجم','مل','لتر','قطعه','حبه','كيس','ظرف','فتله','بكره',
             'خصم','قطعة','عبوة','حبة','علبة','صغير','كبير','سعر','ع','ق'}

def find_brand_in_bs_text(text_norm):
    """Detect MaxAB brand name in BS product text."""
    if not text_norm:
        return []
    tokens = text_norm.split()
    if not tokens:
        return []
    
    results = []
    max_start = 1 if tokens[0] in CATEGORY_WORDS else 0
    
    for brand in maxab_brands_by_length:
        bt = brand.split()
        n = len(bt)
        if n == 1 and len(bt[0]) <= 2:
            continue
        for start in range(max_start + 1):
            if start + n > len(tokens):
                continue
            seg = tokens[start:start+n]
            if all(b == s or strip_al(b) == strip_al(s) for b, s in zip(bt, seg)):
                results.append((brand, 100 if start == 0 else 95))
                break
            if n >= 2 and start == 0:
                if fuzz.ratio(bt[0], tokens[0]) >= 85:
                    if all(any(fuzz.ratio(b, tokens[i]) >= 85 for i in range(min(n+1, len(tokens)))) for b in bt):
                        results.append((brand, 90))
                        break
    
    if not results:
        for pos in range(max_start + 1):
            if pos < len(tokens) and len(tokens[pos]) >= 3 and tokens[pos] in maxab_brand_set:
                results.append((tokens[pos], 90 if pos == 0 else 85))
                break
    
    if len(results) > 1:
        bts = [(b, set(b.split()), s) for b, s in results]
        f = [(b, s) for b, bt, s in bts if not any(bt < o for _, o, _ in bts if o != bt)]
        results = f if f else results
    
    return results[:3]

bs['detected_brands'] = bs['sku_norm'].apply(find_brand_in_bs_text)
bs['top_brand'] = bs['detected_brands'].apply(lambda x: x[0][0] if x else None)
bs['brand_score'] = bs['detected_brands'].apply(lambda x: x[0][1] if x else 0)

detected = bs['top_brand'].notna().sum()
print(f"Brand detection: {detected}/{len(bs)} ({100*detected/len(bs):.1f}%) BS SKUs matched to a MaxAB brand")
print(f"\nTop 20 detected brands:")
print(bs[bs['top_brand'].notna()]['top_brand'].value_counts().head(20).to_string())

In [ ]:
VARIANT_WORDS = {
    'برتقال','تفاح','مانجو','جوافه','عنب','فراوله','موز','اناناس','كوكتيل',
    'ليمون','كريز','توت','خوخ','مشمش','رمان','كمثري','بطيخ','تين',
    'كراميل','فانيليا','شوكولاته','شيكولاته','بندق','فول سوداني','جوز الهند',
    'لافندر','صنوبر','ورد','ياسمين','مسك','عود','عنبر',
    'اخضر','احمر','ازرق','اصفر','ابيض','اسود','وردي','موف','فضي','ذهبي','رمادي','زهري',
    'شيدر','رومي','فيتا','موتزاريلا','بسطرمه','قريش','جولد','استامبولي',
    'سلامي','هوت دوج','برجر','بسطرمه','لانشون','بيفي',
    'مصاصه','مارشملو','اكلير','لولي','ويفر','بسكويت','كرواسون','كيك',
    'فويل','الومنيوم','اكياس','قمامه','ارضيات','زجاج','حمام','مطبخ',
    'حلاوه','طحينه','عسل','مربي',
    'شعريه','خواتم','اسباجيتي','لسان','عصفور','هلاليه','مرمريه','لازانيا','فوسيلي','بنه',
    'نعناع','يانسون','كركديه','شاي','قرفه',
    'سائل','بودر','جيل','سبراي','كريم','صابون','رول','مناديل',
    'كمون','فلفل','كاري','بابريكا','زعتر','كركم',
    'شاور','زنجبيل','جنزبيل','اقراص',
}
VARIANT_WORDS = {normalize_arabic(w) for w in VARIANT_WORDS}

def find_variants_in_text(tokens, brand_tokens):
    found = set()
    for t in tokens:
        if t in brand_tokens: continue
        t_stripped = strip_al(t)
        if t in VARIANT_WORDS: found.add(t)
        elif t_stripped in VARIANT_WORDS: found.add(t_stripped)
    return found

def variant_conflict(sn, mn, bn):
    """Check if two products have conflicting variant/flavor words."""
    ts, tm = sn.split(), mn.split()
    bt = set(bn.split()) if bn else set()
    vs, vm = find_variants_in_text(ts, bt), find_variants_in_text(tm, bt)
    if not vs or not vm: return None
    if vs & vm: return False
    return True

NOISE_TOKENS = STOPWORDS | {normalize_arabic(w) for w in [
    'بطعم','برائحه','محشو','بكريمه','مغطس','مغلف',
    'خاص','سعر','ع','ق','متر','باكو','باكت','جرام','كيلو',
    'بلاستيك','زجاج','حديد','صفيح','نص','ربع']}

def get_descriptive_tokens(text_norm, brand_norm):
    bt = set(brand_norm.split()) if brand_norm else set()
    return set(t for t in text_norm.split() if t not in bt and t not in NOISE_TOKENS and not t.isdigit() and len(t) >= 3 and not re.match(r'^\\d+', t))

def descriptive_overlap(sn, mn, bn):
    """Check if non-brand descriptive tokens overlap between two SKUs."""
    ds, dm = get_descriptive_tokens(sn, bn), get_descriptive_tokens(mn, bn)
    if not ds or not dm: return None
    stripped_ds = {strip_al(t) for t in ds}
    stripped_dm = {strip_al(t) for t in dm}
    if stripped_ds & stripped_dm: return True
    for st in stripped_ds:
        for mt in stripped_dm:
            if fuzz.ratio(st, mt) >= 80: return True
    return False

def price_compatible_bs(bs_row, maxab_row):
    """Check price compatibility between BS and MaxAB products.
    Tries multiple comparison strategies accounting for packing unit differences."""
    bs_total = bs_row['BS_PRICE']
    bs_units = bs_row['BS_BASIC_UNIT_COUNT']
    bs_per_unit = bs_total / bs_units if bs_units > 0 else bs_total
    
    mx_price = maxab_row['CURRENT_PRICE']
    mx_buc = maxab_row['BASIC_UNIT_COUNT']
    mx_child = maxab_row['CHILD_QUANTITY']
    mx_single = mx_price / mx_child if mx_child > 0 else mx_price
    
    comparisons = []
    
    # Per-single-unit price comparison (BS unit price vs MaxAB single unit price)
    # Only valid when the total-to-unit ratio is consistent
    if mx_single > 0 and bs_per_unit > 0:
        unit_diff = pct_diff(bs_per_unit, mx_single)
        # Sanity: total price ratio should be close to unit count ratio
        if bs_units > 0 and mx_child > 0:
            expected_total_ratio = bs_units / mx_child
            actual_total_ratio = bs_total / mx_price if mx_price > 0 else 0
            ratio_check = abs(actual_total_ratio - expected_total_ratio) / max(expected_total_ratio, 0.01) < 0.25
        else:
            ratio_check = True
        if ratio_check:
            comparisons.append((unit_diff, 'per_single'))
    
    # Direct total price comparison (same package level)
    if mx_price > 0:
        comparisons.append((pct_diff(bs_total, mx_price), 'direct'))
    
    # Multiplier: BS packs differently than MaxAB (e.g. BS carton of 12 vs MaxAB single)
    # Only valid when sizes already match (enforced by candidate filter)
    if mx_single > 0 and bs_per_unit > 0:
        ratio = bs_per_unit / mx_single
        nearest_int = round(ratio)
        if 1 < nearest_int <= 50 and nearest_int % 2 == 0 and abs(ratio - nearest_int) / nearest_int < 0.10:
            adjusted = bs_per_unit / nearest_int
            comparisons.append((pct_diff(adjusted, mx_single), f'multiplier_{nearest_int}'))
    
    if not comparisons:
        return False, 1.0, 'none'
    
    best = min(comparisons, key=lambda x: x[0])
    return best[0] <= 0.20, best[0], best[1]

def get_candidates(bs_row, top_n=10):
    """Find candidate MaxAB products for a BS product."""
    bn = bs_row.get('top_brand')
    sn = bs_row['sku_norm']
    bs_cat = bs_row['cat_norm']
    
    if not bn:
        return []
    
    cids = set()
    if bn in brand_to_products:
        cids = brand_to_products[bn]
    
    if not cids:
        return []
    
    scored = []
    for pid in cids:
        if pid not in maxab_products.index:
            continue
        row = maxab_products.loc[pid]
        
        if not category_compatible(bs_cat, row['cat_norm']):
            continue
        
        sz = sizes_compatible(sn, row['sku_norm'], 0.15)
        if sz is False:
            continue
        # BS is 1-to-1 mapping: if both have size info, they must match
        bs_has_size = bool(extract_size_info(sn))
        mx_has_size = bool(extract_size_info(row['sku_norm']))
        if bs_has_size and mx_has_size and sz is not True:
            continue
        
        vc = variant_conflict(sn, row['sku_norm'], row['brand_norm'])
        if vc is True:
            continue
        
        ts = text_sim(sn, row['sku_norm'])
        
        # For lower similarity, require descriptive token overlap
        if ts < 90:
            desc_ok = descriptive_overlap(sn, row['sku_norm'], row['brand_norm'])
            if desc_ok is False:
                continue
        
        score = ts + (10 if sz is True else 0)
        scored.append((pid, score, ts))
    
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_n]

def map_one_bs(bs_row, min_ts=40):
    """Map a single BS product to MaxAB products."""
    cands = get_candidates(bs_row)
    if not cands:
        return []
    
    matches = []
    for pid, cs, ts in cands:
        if ts < min_ts:
            continue
        
        mx_rows = maxab[maxab['PRODUCT_ID'] == pid]
        for _, mx in mx_rows.iterrows():
            po, pd_, ps = price_compatible_bs(bs_row, mx)
            if pd_ > 0.20:
                continue
            
            price_score = max(0, 30 * (1 - pd_)) if po else 0
            combined = cs + price_score
            
            matches.append({
                'PRODUCT_ID': mx['PRODUCT_ID'],
                'SKU': mx['SKU'],
                'BRAND': mx['BRAND'],
                'CAT': mx['CAT'],
                'MAXAB_PRICE': mx['CURRENT_PRICE'],
                'MAXAB_BUC': mx['BASIC_UNIT_COUNT'],
                'MAXAB_CHILD_QTY': mx['CHILD_QUANTITY'],
                'MAXAB_SINGLE_PRICE': mx['single_unit_price'],
                'PACKING_UNIT_NAME_AR': mx['PACKING_UNIT_NAME_AR'],
                'text_similarity': ts,
                'price_compatible': po,
                'price_diff_pct': pd_,
                'price_strategy': ps,
                'combined_score': combined,
                'BEN_SOLIMAN_SKU': bs_row['BEN_SOLIMAN_SKU'],
                'BS_CATEGORY': bs_row['CATEGORY'],
                'BS_PACKAGE_UNIT': bs_row['BS_PACKAGE_UNIT'],
                'BS_BASIC_UNIT_COUNT': bs_row['BS_BASIC_UNIT_COUNT'],
                'BS_PRICE': bs_row['BS_PRICE'],
                'BS_UNIT_PRICE': bs_row['unit_price'],
                'BS_MATCHED_PRICE': round(
                    bs_row['BS_PRICE'] if ps == 'direct'
                    else (bs_row['unit_price'] / int(ps.split('_')[1]) * mx['CHILD_QUANTITY']) if ps.startswith('multiplier_')
                    else bs_row['unit_price'] * mx['CHILD_QUANTITY'],
                    2),
                'detected_brand': bs_row.get('top_brand', ''),
                'match_source': 'algorithm',
            })
    
    matches.sort(key=lambda x: x['combined_score'], reverse=True)
    matches = [m for m in matches if m['combined_score'] >= 100]
    return matches

print("Matching functions loaded.")

In [ ]:
print(f"Running algorithm on {len(bs)} BS products...")
print(f"Started at {datetime.now().strftime('%H:%M:%S')}")

results = []
unmatched = []
mc = {'t': 0, 'm': 0}

for idx, row in bs.iterrows():
    ms = map_one_bs(row)
    mc['t'] += 1
    if ms:
        results.append(ms[0])  # 1-to-1: only keep the single best match per BS SKU
        mc['m'] += 1
    else:
        unmatched.append({
            'BEN_SOLIMAN_SKU': row['BEN_SOLIMAN_SKU'],
            'BS_CATEGORY': row['CATEGORY'],
            'BS_PACKAGE_UNIT': row['BS_PACKAGE_UNIT'],
            'BS_PRICE': row['BS_PRICE'],
            'detected_brand': row.get('top_brand', ''),
        })
    if mc['t'] % 500 == 0:
        print(f"  {mc['t']}/{len(bs)} ({mc['m']} matched)")

df_results = pd.DataFrame(results)
df_unmatched = pd.DataFrame(unmatched)

# Deduplicate: 1 BS SKU → 1 MaxAB product (best score wins)
if len(df_results) > 0:
    df_results = df_results.sort_values('combined_score', ascending=False).drop_duplicates(
        subset=['BEN_SOLIMAN_SKU'], keep='first'
    ).reset_index(drop=True)

# Deduplicate: 1 MaxAB product → 1 BS SKU (highest combined_score wins)
if len(df_results) > 0:
    before_maxab_dedup = len(df_results)
    df_results = df_results.sort_values('combined_score', ascending=False).drop_duplicates(
        subset=['PRODUCT_ID'], keep='first'
    ).reset_index(drop=True)
    print(f"MaxAB dedup: {before_maxab_dedup} → {len(df_results)} (removed {before_maxab_dedup - len(df_results)} duplicate MaxAB mappings)")

print(f"\nDone at {datetime.now().strftime('%H:%M:%S')}")
print(f"Matched: {mc['m']}/{mc['t']} ({100*mc['m']/max(mc['t'],1):.1f}%)")
print(f"Result rows: {len(df_results)}")
print(f"Unmatched: {len(df_unmatched)}")

In [ ]:
if len(df_results) > 0:
    our_matches = set(zip(df_results['BEN_SOLIMAN_SKU'], df_results['PRODUCT_ID']))
    print(f"\nOur unique matches: {len(our_matches)}")
    
    if len(savvy) > 0:
        savvy_matches = set(zip(savvy['BS_DESC'], savvy['MAXAB_PRODUCT_ID']))
        overlap = len(our_matches & savvy_matches)
        print(f"Savvy matches: {len(savvy_matches)}")
        print(f"Overlap: {overlap}")
        print(f"Only in ours: {len(our_matches - savvy_matches)}")
        print(f"Only in Savvy: {len(savvy_matches - our_matches)}")

In [ ]:
if len(df_results) > 0:
    df_results.to_excel('bs_mapping_results.xlsx', index=False)
    print(f"Results exported: {len(df_results)} rows")

if len(df_unmatched) > 0:
    df_unmatched.to_excel('bs_mapping_unmatched.xlsx', index=False)
    print(f"Unmatched exported: {len(df_unmatched)} rows")

print(f"\n=== SUMMARY ===")
print(f"BS products: {len(bs)}")
print(f"Matched: {mc['m']} ({100*mc['m']/len(bs):.1f}%)")
print(f"Unmatched: {len(df_unmatched)} ({100*len(df_unmatched)/len(bs):.1f}%)")
if len(df_results) > 0:
    print(f"Unique MaxAB products matched: {df_results['PRODUCT_ID'].nunique()}")
    print(f"\nPrice compatibility distribution:")
    print(df_results['price_strategy'].value_counts().to_string())
    print(f"\nAvg text similarity: {df_results['text_similarity'].mean():.1f}")
    print(f"Avg price diff: {df_results['price_diff_pct'].mean():.2%}")

In [ ]:
if len(df_results) > 0:
    original_columns = df_results.columns.tolist()

    status = upload_dataframe_to_snowflake(
        "Egypt", df_results, "MATERIALIZED_VIEWS", "bensoliman_inhouse_mapping", "overwrite"
    )
    print(f"Snowflake upload status: {status}")

    df_results.columns = original_columns

    SLACK_CHANNEL_ID = 'C0AAWK97Z3Q'
    send_file_slack(
        df_results,
        f"BS Mapping Update — {datetime.now().strftime('%Y-%m-%d %H:%M')} | {len(df_results)} matches",
        SLACK_CHANNEL_ID,
        filename='bs_mapping_results.xlsx'
    )
else:
    print("No results to upload.")